In [ ]:
import nltk
nltk.download('averaged_perceptron_tagger_eng')

In [ ]:
LOCAL_BERT_PATH = r"/kaggle/input/bert_model/transformers/default/1/bert_model"

In [ ]:
import pandas as pd
import re
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from nltk.tokenize import TweetTokenizer
import nltk
from tqdm import tqdm



In [ ]:
# 缩写转换字典
ABBREVIATIONS = {
   "he's": "he is", 
   "there's": "there is", 
   "We're": "We are", 
   "That's": "That is", 
   "won't": "will not", 
   "they're": "they are", 
   "Can't": "Can not", 
   "wasn't": "was not", 
   "don't": "do not", 
   "aren't": "are not", 
   "isn't": "is not", 
   "What's": "What is", 
   "haven't": "have not", 
   "hasn't": "has not", 
   "There's": "There is", 
   "He's": "He is", 
   "It's": "It is", 
   "You're": "You are", 
   "I'M":"I am", 
   "shouldn't": "should not", 
   "wouldn't": "would not", 
   "i'm": "I am", 
   "I'm": "I am", 
   "Isn't": "is not", 
   "Here's":"Here is", 
   "you've": "you have", 
   "we're": "we are", 
   "what's":"what is", 
   "couldn't": "could not", 
   "we've": "we have", 
   "it's": "it is", 
   "doesn't": "does not", 
   "It's": "It is", 
   "who's": "who is", 
   "I've": "I have", 
   "y'all": "you all", 
   "would've": "would have", 
   "it'll": "it will", 
   "we'll": "we will", 
   "We've": "We have", 
   "he'll": "he will", 
   "Y'all": "You all", 
   "Weren't": "Were not", 
   "Didn't": "Did not", 
   "they'll": "they will", 
   "they'd": "they would", 
   "DON'T": "DO NOT", 
   "That's": "That is", 
   "they've": "they have", 
   "i'd": "I would", 
   "should've": "should have", 
   "You're": "You are", 
   "where's": "where is",
   "we'd": "we would", 
   "i'll": "I will", 
   "weren't": "were not", 
   "They're": "They are", 
   "you'll": "you will", 
   "I'd": "I would", 
   "let's": "let us", 
   "it's": "it is", 
   "can't": "can not", 
   "don't": "do not", 
   "you're": "you are", 
   "i've": "I have", 
   "that's": "that is", 
   "i'll": "I will", 
   "doesn't": "does not", 
   "i'd": "I would", 
   "didn't": "did not", 
   "ain't": "am not", 
   "I've": "I have", 
   "Don't": "do not", 
   "I'll": "I will", 
   "I'd":"I would", 
   "Let's": "Let us", 
   "you'd": "You would", 
   "It's": "It is", 
   "Ain't": "am not", 
   "Haven't": "Have not", 
   "Could've": "Could have", 
   "youve": "you have",   
   "w/": "with",
    "w/o": "without",
    "b/c": "because",
    "thx": "thanks",
    "u": "you",
    "r": "are",
    "rt": "retweet"
}

In [ ]:
from nltk import pos_tag
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet
from emoji import demojize
from textblob import TextBlob
from nltk.corpus import stopwords


class CustomTweetCleaner:
    def __init__(self):
        self.tokenizer = TweetTokenizer(preserve_case=False, reduce_len=True, strip_handles=True)
        self.lemmatizer = WordNetLemmatizer()

    def get_wordnet_pos(self, treebank_tag):
        """将NLTK的POS标签转换为WordNet的POS标签"""
        if treebank_tag.startswith('J'):
            return wordnet.ADJ
        elif treebank_tag.startswith('V'):
            return wordnet.VERB
        elif treebank_tag.startswith('N'):
            return wordnet.NOUN
        elif treebank_tag.startswith('R'):
            return wordnet.ADV
        else:
            return wordnet.NOUN  # 默认使用名词

    def clean_text(self, text):
        # 初始化停用词列表
        stop_words = set(stopwords.words('english'))
        
        text = demojize(text)  # 转换表情符号为文本
        text = re.sub(r'#(\w+)', r'\1', text)  # 移除标签中的 #
        # 现有清洗步骤（URL、缩写、标点）
        text = re.sub(r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\\(\\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+', '', text)
        for abbr, full in ABBREVIATIONS.items():
            text = re.sub(r'\b' + re.escape(abbr) + r'\b', full, text, flags=re.IGNORECASE)
        text = re.sub(r'[^\w\s]', '', text)
        tokens = self.tokenizer.tokenize(text)
        tagged_tokens = pos_tag(tokens)
        # 词形还原并移除停用词
        lemmatized_tokens = [
            self.lemmatizer.lemmatize(token, self.get_wordnet_pos(pos))
            for token, pos in tagged_tokens
            if token.lower() not in stop_words  # 移除停用词
        ]
        return ' '.join(lemmatized_tokens).strip()
    
    def tokenize(self, text):
        """自定义分词器"""
        cleaned_text = self.clean_text(text)
        tokens = self.tokenizer.tokenize(cleaned_text)
        return tokens

In [ ]:
!pip install nlpaug

In [ ]:
import pandas as pd
import nlpaug.augmenter.word as naw



def augment_data(data, target_size=10000):
    # 初始化数据增强器
    aug = naw.SynonymAug(aug_src='wordnet', aug_p=0.3)
    
    # 当前数据量和需要增加的数量
    current_size = len(data)
    samples_needed = target_size - current_size

    if samples_needed <= 0:
        print("No augmentation needed.")
        return data

    # 类别分布
    class_counts = data['target'].value_counts()
    minority_class = 1 if class_counts.get(1, 0) < class_counts.get(0, 0) else 0
    minority_samples = int(samples_needed * 0.7)
    majority_samples = samples_needed - minority_samples

    augmented_texts = []
    augmented_targets = []

    # 增强少数类
    minority_data = data[data['target'] == minority_class]
    for _ in range(minority_samples):
        text = minority_data['text'].sample(1).iloc[0]
        aug_text = aug.augment(text)[0]
        augmented_texts.append(aug_text)
        augmented_targets.append(minority_class)

    # 增强多数类
    majority_data = data[data['target'] != minority_class]
    for _ in range(majority_samples):
        text = majority_data['text'].sample(1).iloc[0]
        aug_text = aug.augment(text)[0]
        augmented_texts.append(aug_text)
        augmented_targets.append(1 - minority_class)

    # 创建增强数据 DataFrame，保持原始列结构（除 text, target 外置为 None）
    empty_columns = {col: [None] * samples_needed for col in data.columns if col not in ['text', 'target']}
    augmented_df = pd.DataFrame({
        'text': augmented_texts,
        'target': augmented_targets,
        **empty_columns
    })

    # 拼接回原始数据
    final_data = pd.concat([data, augmented_df], ignore_index=True)

    # 打印结果
    print(f"Data size after augmentation: {len(final_data)}")
    print(f"Class distribution after augmentation:\n{final_data['target'].value_counts()}")

    return final_data

使用bert

In [ ]:
import pandas as pd
import re
import numpy as np
import torch
print(torch.__version__)
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertForSequenceClassification
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report
from sklearn.model_selection import train_test_split,StratifiedKFold
from nltk.tokenize import TweetTokenizer
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet
from nltk import pos_tag
import optuna
import nltk
from tqdm import tqdm
import matplotlib.pyplot as plt
from transformers import get_linear_schedule_with_warmup

In [ ]:
# 自定义数据集类
class TweetDataset(Dataset):
    def __init__(self, texts, keywords, labels, tokenizer, max_len=128):
        self.texts = texts
        self.keywords = keywords
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.cleaner = CustomTweetCleaner()

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        keyword = self.keywords[idx] if self.keywords[idx] else 'none'
        label = self.labels[idx]

        # 清理文本和keyword
        cleaned_text = self.cleaner.clean_text(text)
        cleaned_keyword = self.cleaner.clean_text(keyword)

        # 拼接keyword和text
        input_text = f"{cleaned_keyword} [SEP] {cleaned_text}"

        # 编码输入
        encoding = self.tokenizer.encode_plus(
            input_text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

In [ ]:
def plot_classification_metrics(epochs, val_accuracies, val_f1_scores, val_precisions, val_recalls,batch_size, learning_rate, fold=None):
    """绘制分类报告曲线：Accuracy, F1, Precision, Recall"""
    plt.figure(figsize=(10, 6))
    plt.plot(range(1, epochs + 1), val_accuracies, label='Val Accuracy', color='blue', marker='o')
    plt.plot(range(1, epochs + 1), val_f1_scores, label='Val F1 Score', color='gold', marker='o')
    plt.plot(range(1, epochs + 1), val_precisions, label='Val Precision', color='cyan', marker='o')
    plt.plot(range(1, epochs + 1), val_recalls, label='Val Recall', color='purple', marker='o')
    plt.xlabel('Epoch')
    plt.ylabel('Metric Value')
    lr_str = f"{learning_rate:.0e}"  # 科学计数法表示
    if fold is not None:
        plt.title(f'Fold {fold} Metrics (bs={batch_size}, lr={lr_str})')
        # filename = f'picture/classification_metrics_fold_{fold}_bs{batch_size}_lr{lr_str}.png'
    else:
        plt.title(f'CV Metrics (bs={batch_size}, lr={lr_str})')
        # filename = f'picture/classification_metrics_cv_bs{batch_size}_lr{lr_str}.png'
    plt.legend()
    plt.grid(True)
    plt.ylim(0, 1)
    plt.xticks(range(1, epochs + 1))
    # plt.savefig(filename)
    plt.show()
    plt.close()

def plot_loss_curves(epochs, train_losses, val_losses,batch_size, learning_rate, fold=None):
    """绘制损耗曲线：Train Loss, Val Loss"""
    plt.figure(figsize=(10, 6))
    plt.plot(range(1, epochs + 1), train_losses, label='Train Loss', color='cyan', marker='o')
    plt.plot(range(1, epochs + 1), val_losses, label='Val Loss', color='red', marker='o')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    lr_str = f"{learning_rate:.0e}"
    if fold is not None:
        plt.title(f'Fold {fold} Loss (bs={batch_size}, lr={lr_str})')
        # filename = f'picture/loss_curves_fold_{fold}_bs{batch_size}_lr{lr_str}.png'
    else:
        plt.title(f'CV Loss (bs={batch_size}, lr={lr_str})')
        # filename = f'picture/loss_curves_cv_bs{batch_size}_lr{lr_str}.png'
    plt.legend()
    plt.grid(True)
    plt.ylim(0, max(max(train_losses), max(val_losses)) * 1.1)
    plt.xticks(range(1, epochs + 1))
    # plt.savefig(filename)
    plt.show()
    plt.close()

In [ ]:
def compute_class_weights(data):
    """Calculate class weights based on inverse class frequency."""
    class_counts = data['target'].value_counts()
    total_samples = len(data)
    class_weights = {}
    
    # Inverse frequency: weight = total_samples / (n_classes * class_count)
    for target in class_counts.index:
        class_weights[target] = total_samples / (2 * class_counts[target])
    
    # Convert to tensor for use in loss function
    weights = [class_weights[0], class_weights[1]]
    return torch.tensor(weights, dtype=torch.float)

In [ ]:
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts

def train_bert_model(data, model_name, epochs=10, batch_size=16, max_len=128, learning_rate=2e-5, n_splits=3):
    # cleaner = CustomTweetCleaner()
    # data['text_cleaned'] = data['text'].apply(cleaner.clean_text)
    # data['keyword'] = data['keyword'].fillna('none').apply(cleaner.clean_text)
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    X = data['text'].values
    X_keyword = data['keyword'].values
    y = data['target'].values

    # Compute class weights
    class_weights = compute_class_weights(data)
    print(f"Class weights: {class_weights}")

    to_train_loss = 0
    to_val_loss = 0
    to_val_accuracy = 0
    to_val_f1 = 0
    to_val_precision = 0
    to_val_recall = 0

    for fold, (trn_idx, val_idx) in enumerate(skf.split(X, y), 1):
        print(f'\nFold {fold}/{n_splits}')
        print(f'Fold {fold} Training Set Shape = {data.loc[trn_idx, "text"].shape} - Validation Set Shape = {data.loc[val_idx, "text"].shape}')
        print(f'Fold {fold} Training Set Unique keyword Count = {data.loc[trn_idx, "keyword"].nunique()} - Validation Set Unique keyword Count = {data.loc[val_idx, "keyword"].nunique()}')

        X_train = X[trn_idx].tolist()
        X_val = X[val_idx].tolist()
        X_train_keyword = X_keyword[trn_idx].tolist()
        X_val_keyword = X_keyword[val_idx].tolist()
        y_train = data['target'].values[trn_idx].tolist()
        y_val = data['target'].values[val_idx].tolist()

        try:
            tokenizer = BertTokenizer.from_pretrained(model_name)
            model = BertForSequenceClassification.from_pretrained(model_name, num_labels=2, hidden_dropout_prob=0.2)
            # 只训练最后两层
            # for name, param in model.bert.named_parameters():
            #     if not (name.startswith("encoder.layer.9") or 
            #             name.startswith("encoder.layer.10") or 
            #             name.startswith("encoder.layer.11")):
            #         param.requires_grad = False
        except Exception as e:
            print(f"Error loading BERT model or tokenizer: {e}")
            raise

        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        print(f"Using device: {device}")
        model.to(device)
        class_weights = class_weights.to(device)  # Move weights to the same device as the model

        train_dataset = TweetDataset(X_train, X_train_keyword, y_train, tokenizer, max_len)
        val_dataset = TweetDataset(X_val, X_val_keyword, y_val, tokenizer, max_len)
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=batch_size)

        optimizer = AdamW(model.parameters(), lr=learning_rate, weight_decay=0.05)
        # total_steps = len(train_loader) * epochs
        # scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0.1 * total_steps, num_training_steps=total_steps)
        scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=2, T_mult=2, eta_min=1e-6)

        # Define weighted loss function
        loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)

        best_f1 = 0
        patience = 2
        early_stop_counter = 0

        fold_train_losses = []
        fold_val_losses = []
        fold_val_accuracies = []
        fold_val_f1_scores = []
        fold_val_precisions = []
        fold_val_recalls = []

        for epoch in range(epochs):
            model.train()
            total_train_loss = 0
            for batch in tqdm(train_loader, desc=f"Fold {fold} Epoch {epoch + 1} Training"):
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['labels'].to(device)
                optimizer.zero_grad()
                outputs = model(input_ids, attention_mask=attention_mask)
                logits = outputs.logits
                loss = loss_fn(logits, labels)  # Use weighted loss
                total_train_loss += loss.item()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                scheduler.step()

            avg_train_loss = total_train_loss / len(train_loader)
            fold_train_losses.append(avg_train_loss)

            model.eval()
            total_val_loss = 0
            predictions = []
            true_labels = []
            with torch.no_grad():
                for batch in tqdm(val_loader, desc=f"Fold {fold} Epoch {epoch + 1} Validation"):
                    input_ids = batch['input_ids'].to(device)
                    attention_mask = batch['attention_mask'].to(device)
                    labels = batch['labels'].to(device)
                    outputs = model(input_ids, attention_mask=attention_mask)
                    logits = outputs.logits
                    loss = loss_fn(logits, labels)  # Use weighted loss for validation
                    total_val_loss += loss.item()
                    preds = torch.argmax(logits, dim=1)
                    predictions.extend(preds.cpu().numpy())
                    true_labels.extend(labels.cpu().numpy())

            avg_val_loss = total_val_loss / len(val_loader)
            val_accuracy = accuracy_score(true_labels, predictions)
            val_f1 = f1_score(true_labels, predictions, average='weighted')
            val_precision = precision_score(true_labels, predictions, average='weighted')
            val_recall = recall_score(true_labels, predictions, average='weighted')

            fold_val_losses.append(avg_val_loss)
            fold_val_accuracies.append(val_accuracy)
            fold_val_f1_scores.append(val_f1)
            fold_val_precisions.append(val_precision)
            fold_val_recalls.append(val_recall)

            print(f"Fold {fold} Epoch {epoch + 1}:")
            print(f"  Train Loss: {avg_train_loss:.4f}")
            print(f"  Val Loss: {avg_val_loss:.4f}")
            print(f"  Val Accuracy: {val_accuracy:.4f}")
            print(f"  Val F1 Score: {val_f1:.4f}")
            print(f"  Val Precision: {val_precision:.4f}")
            print(f"  Val Recall: {val_recall:.4f}")

            scheduler.step(avg_val_loss)

            if val_f1 > best_f1:
                best_f1 = val_f1
                early_stop_counter = 0
            else:
                early_stop_counter += 1
                if early_stop_counter >= patience:
                    print(f"Early stopping triggered at epoch {epoch + 1}")
                    break

        plot_classification_metrics(len(fold_train_losses), fold_val_accuracies, fold_val_f1_scores, fold_val_precisions, fold_val_recalls, batch_size, learning_rate, fold=fold)
        plot_loss_curves(len(fold_train_losses), fold_train_losses, fold_val_losses, batch_size, learning_rate, fold=fold)

        print(f"\nFold {fold} Final Evaluation:")
        print(f"Accuracy: {val_accuracy}")
        print(f"F1 Score: {val_f1}")
        print(f"Precision: {val_precision}")
        print(f"Recall: {val_recall}")
        print("\nClassification Report:\n", classification_report(true_labels, predictions))

        to_train_loss += fold_train_losses[-1]
        to_val_loss += fold_val_losses[-1]
        to_val_accuracy += fold_val_accuracies[-1]
        to_val_f1 += fold_val_f1_scores[-1]
        to_val_precision += fold_val_precisions[-1]
        to_val_recall += fold_val_recalls[-1]

    av_train_loss = to_train_loss / n_splits
    av_val_loss = to_val_loss / n_splits
    av_val_accuracy = to_val_accuracy / n_splits
    av_val_f1_score = to_val_f1 / n_splits
    av_val_precision = to_val_precision / n_splits
    av_val_recall = to_val_recall / n_splits

    # 打印最终平均指标
    print("\nCross-Validation Final Average Metrics (last epoch per fold):")
    print(f"  Avg Train Loss: {av_train_loss:.4f}")
    print(f"  Avg Val Loss: {av_val_loss:.4f}")
    print(f"  Avg Val Accuracy: {av_val_accuracy:.4f}")
    print(f"  Avg Val F1 Score: {av_val_f1_score:.4f}")
    print(f"  Avg Val Precision: {av_val_precision:.4f}")
    print(f"  Avg Val Recall: {av_val_recall:.4f}")

    return av_val_f1_score

In [ ]:
def train_final_bert_model(data, model_name, epochs=10, batch_size=32, max_len=128, learning_rate=2e-5):
    # cleaner = CustomTweetCleaner()
    # data['text_cleaned'] = data['text'].apply(cleaner.clean_text)
    # data['keyword'] = data['keyword'].fillna('none').apply(cleaner.clean_text)
    texts = data['text'].tolist()
    keywords = data['keyword'].tolist()
    targets = data['target'].tolist()

    # Compute class weights
    class_weights = compute_class_weights(data)
    print(f"Class weights: {class_weights}")

    tokenizer = BertTokenizer.from_pretrained(model_name)
    model = BertForSequenceClassification.from_pretrained(model_name, num_labels=2, hidden_dropout_prob=0.2)
    # # 只训练最后两层
    # for name, param in model.bert.named_parameters():
    #     if not (name.startswith("encoder.layer.9") or 
    #             name.startswith("encoder.layer.10") or 
    #             name.startswith("encoder.layer.11")):
    #         param.requires_grad = False


    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    class_weights = class_weights.to(device)  # Move weights to device

    full_dataset = TweetDataset(texts, keywords, targets, tokenizer, max_len)
    full_loader = DataLoader(full_dataset, batch_size=batch_size, shuffle=True)

    optimizer = AdamW(model.parameters(), lr=learning_rate, weight_decay=0.05)
    # total_steps = len(train_loader) * epochs
    # scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0.1 * total_steps, num_training_steps=total_steps)
    scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=2, T_mult=2, eta_min=1e-6)

    # Define weighted loss function
    loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)

    avg_losses = []
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for batch in tqdm(full_loader, desc=f"Final Training Epoch {epoch + 1}"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            optimizer.zero_grad()
            outputs = model(input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            loss = loss_fn(logits, labels)  # Use weighted loss
            total_loss += loss.item()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()

        avg_loss = total_loss / len(full_loader)
        avg_losses.append(avg_loss)
        print(f"Epoch {epoch + 1} - Avg Training Loss: {avg_loss:.4f}")

    plt.figure(figsize=(8, 5))
    plt.plot(range(1, epochs + 1), avg_losses, marker='o', color='red', label='Avg Training Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title(f'Final Model Training Loss Curve (bs={batch_size}, lr={learning_rate:.0e})')
    plt.grid(True)
    plt.xticks(range(1, epochs + 1))
    plt.legend()
    plt.tight_layout()
    # plt.savefig('picture/final_training_loss.png')
    plt.show()
    plt.close()

    return model, tokenizer

In [ ]:
def objective(trial, data, model_name=LOCAL_BERT_PATH, max_len=128, n_splits=3):
    """Optuna目标函数，使用基于keyword的交叉验证优化超参数"""
    # 定义搜索空间
    learning_rate = trial.suggest_categorical('learning_rate', [ 2e-5])
    batch_size = trial.suggest_categorical('batch_size', [32])
    epochs = 10

    print(f"  learning_rate: {learning_rate}")
    print(f"  batch_size: {batch_size}")

    # 训练模型并返回平均F1分数
    avg_f1 = train_bert_model(
        data,
        model_name=model_name,
        epochs=epochs,
        batch_size=batch_size,
        max_len=max_len,
        learning_rate=learning_rate,
        n_splits=n_splits
    )
    return avg_f1

In [ ]:
from optuna.samplers import GridSampler
def optimize_hyperparameters(data, model_name=LOCAL_BERT_PATH, n_trials=4, n_splits=3):
    """使用Optuna的GridSampler穷举所有超参数组合"""
    search_space = {
        'learning_rate': [2e-5],
        'batch_size': [32]
    }
    sampler = GridSampler(search_space)

    study = optuna.create_study(direction='maximize', sampler=sampler)
    study.optimize(lambda trial: objective(trial, data, model_name, n_splits=n_splits), n_trials=n_trials)
    
    print("Best hyperparameters:", study.best_params)
    print("Best F1 Score:", study.best_value)
    
    #使用最优超参数重新训练
    best_model, best_tokenizer = train_final_bert_model(
        data,
        model_name=model_name,
        epochs=6,
        batch_size=study.best_params['batch_size'],
        max_len=128,
        learning_rate=study.best_params['learning_rate'],
    )
    
    return best_model, best_tokenizer



In [ ]:
def predict_single_sample_bert(model, tokenizer, text, keyword, max_len=128):
    """预测单条样本"""
    cleaner = CustomTweetCleaner()
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    # print(device)
    
    # # 清理样本
    # cleaned_text = cleaner.clean_text(text)
    # cleaned_keyword = cleaner.clean_text(keyword if keyword else 'none')
    
    # 拼接keyword和text
    input_text = f"{keyword} [SEP] {text}"
    
    # 编码输入
    encoding = tokenizer.encode_plus(
        input_text,
        add_special_tokens=True,
        max_length=max_len,
        padding='max_length',
        truncation=True,
        return_attention_mask=True,
        return_tensors='pt'
    )
    
    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)
    
    # 预测
    model.eval()
    with torch.no_grad():
        outputs = model(input_ids, attention_mask=attention_mask)
        prediction = torch.argmax(outputs.logits, dim=1).cpu().numpy()[0]
    
    return prediction

In [ ]:
def predict_test_set(model, tokenizer, test_data, max_len=128):
    """预测test.csv并生成Kaggle提交文件"""
    cleaner = CustomTweetCleaner()
    
    # 清理测试数据
    test_data['text_cleaned'] = test_data['text'].apply(cleaner.clean_text)
    test_data['keyword'] = test_data['keyword'].fillna('none').apply(cleaner.clean_text)
    
    # 预测
    predictions = []
    for _, row in tqdm(test_data.iterrows(), total=len(test_data), desc="Predicting test set"):
        pred = predict_single_sample_bert(model, tokenizer, row['text'], row['keyword'], max_len)
        predictions.append(pred)
    
    # 生成提交文件
    submission = pd.DataFrame({'id': test_data['id'], 'target': predictions})
    #submission.to_csv('submission.csv', index=False)
    #print("Submission file saved as 'submission.csv'")
    
    return submission

In [ ]:
# 读取train.csv，并赋值给变量train
data = pd.read_csv('/kaggle/input/nlp-getting-started/train.csv')

In [ ]:
# 清洗训练数据
cleaner = CustomTweetCleaner()
data['text'] = data['text'].apply(cleaner.clean_text)
data=augment_data(data)

data['keyword'] = data['keyword'].fillna('none').apply(cleaner.clean_text)

In [ ]:
# 超参数调优并训练最优模型
# model, tokenizer = optimize_hyperparameters(data, n_trials=1)

In [ ]:
model, tokenizer=train_final_bert_model(data, model_name = LOCAL_BERT_PATH, epochs=3, batch_size=32, max_len=128, learning_rate=2e-5)

In [ ]:

test_data = pd.read_csv("/kaggle/input/nlp-getting-started/test.csv")
submission = predict_test_set(model, tokenizer, test_data)

In [ ]:
submission.to_csv('submission.csv', index=False)
print("Submission file saved as 'submission.csv'")